In [113]:
import arcpy
import psycopg
import csv
from dotenv import load_dotenv
import os

In [114]:
def execute_and_append_to_csv(conn, query, output_file, title=None):
    cursor = conn.cursor()

    with conn.transaction():
        cursor.execute(query)

        # Fetch all rows and column names
        rows = cursor.fetchall()
        colnames = [desc[0] for desc in cursor.description]

        # Open the CSV file in append mode
        with open(output_file, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            # Write the title, if provided
            if title:
                writer.writerow([title])  # Write the title row
                writer.writerow([])  # Add a blank line for spacing

            # Add header (column names) row before the data
            writer.writerow(colnames)

            # Write data rows
            writer.writerows(rows)

            # Add a blank line after the data for separation
            writer.writerow([])



        print(f"Data successfully appended with section: {title}")


In [115]:
load_dotenv()

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
sde_path = os.getenv('SDE_PATH')
arcpy.env.workspace = sde_path
schema = os.getenv('SCHEMA')
db = os.getenv('DB_NAME')

print(db)

sweri


In [116]:
    conn = psycopg.connect(
        host=os.getenv('DB_HOST'),
        port=os.getenv('DB_PORT'),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        autocommit=True
    )


In [117]:
# Subset creation for treatments of interest

ti_stats_table = 'treatment_index_fy_17_25_stats'
base_treatment_index = 'treatment_index_analysis'

cursor = conn.cursor()

with conn.transaction():
    cursor.execute(f'''
        DROP TABLE IF EXISTS
        {schema}.{ti_stats_table};
    ''')

    cursor.execute(f'''
        CREATE TABLE 
            {schema}.{ti_stats_table}
        AS
        SELECT * FROM
            {schema}.{base_treatment_index}
        WHERE 
            treatment_date BETWEEN '2016-10-01 00:00:00' AND '2025-09-30 23:59:59'
            and status = 'Completed'
            and identifier_database in ('IFPRS', 'NFPORS', 'FACTS Hazardous Fuels', 'FACTS Common Attributes')
            and (error is null OR error NOT LIKE '%DUPLICATE_DROP%');
    ''')
    

In [118]:
# Treatment Index Calulated Acres for Area Weights

cursor = conn.cursor()

with conn.transaction():
    cursor.execute(f'''
        ALTER TABLE
        	{schema}.{ti_stats_table}
        ADD COLUMN 
        	acres_calc NUMERIC;
    ''')
    
    cursor.execute(f'''
        CREATE INDEX ON 
            {schema}.{ti_stats_table}
        USING GIST(shape);
    ''')
    
    cursor.execute(f'''
        UPDATE
        	{schema}.{ti_stats_table}
        SET	acres_calc = ST_Area(ST_Transform(shape, 5070)) / 4046.8564224;
    ''')

In [119]:
# US States Import

states_service_url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_Census_States/FeatureServer/0"

# Add GACC layers to the map
states_layer = active_map.addDataFromPath(states_service_url)
states_layer.name = 'states_wgs84'

states_fl_name = 'states_wgs84'

# import recent wildfires into postgres
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=states_layer,
    Output_Geodatabase=sde_path
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde'>

In [120]:
ti_state_instersect_layer = 'treatment_index_fy_17_25_stats_intersect_states'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{ti_stats_table}';'{sde_path}\{db}.{schema}.{states_fl_name}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{ti_state_instersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.treatment_index_fy_17_25_stats_intersect_states'>

In [121]:
cursor = conn.cursor()

with conn.transaction():

    # Create and Populate Area Weight
    cursor.execute(f'''
        ALTER TABLE
            {schema}.{ti_state_instersect_layer}
        ADD COLUMN area_weight NUMERIC;
    ''')
    
    cursor.execute(f'''
        UPDATE {schema}.{ti_state_instersect_layer}
        SET area_weight = 1
            WHERE unique_id NOT IN (
                SELECT
                    t.unique_id
                FROM
                    {schema}.{ti_stats_table} t
                JOIN
                    {schema}.{states_fl_name} s
                ON
                    ST_Intersects(t.shape, s.shape)
                GROUP BY
                    t.unique_id
                HAVING
                    COUNT(DISTINCT s.state_abbr) > 1
            );
    ''')
    
    cursor.execute(f'''
        UPDATE 
            {schema}.{ti_state_instersect_layer}
        SET 
            area_weight = (ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) / acres_calc
        WHERE 
            area_weight IS null;
    ''')

    # Use Area Weight to Caluculate Weighted Cost
    cursor.execute(f'''
            ALTER TABLE 
                {schema}.{ti_state_instersect_layer}
            ADD COLUMN
                weighted_cost NUMERIC;
    ''')

    cursor.execute(f'''
        UPDATE {schema}.{ti_state_instersect_layer}
        SET weighted_cost = total_cost * area_weight; 
    ''')

In [122]:
# Historic Wildifires Import

historic_fires_service_url = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/ArcGIS/rest/services/InterAgencyFirePerimeterHistory_All_Years_View/FeatureServer/0"
historic_fires_fl_name = "historic_fires_cy_17_25"

historic_fires_query = "FIRE_YEAR_INT BETWEEN 2017 AND 2025"

arcpy.management.MakeFeatureLayer(historic_fires_service_url, historic_fires_fl_name, where_clause=historic_fires_query)

historic_fires_wgs84 = 'historic_fires_cy_17_25_wgs84'

# reproject to wgs84
arcpy.management.Project(
    in_dataset={historic_fires_wgs84},
    out_dataset=rf"C:\Users\Ben Holloway\Documents\SWERI\ArcPro\Sweri\Sweri.gdb\{historic_fires_wgs84}",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    transform_method=None,
    in_coor_system='PROJCRS["WGS_1984_Web_Mercator_Auxiliary_Sphere",BASEGEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Mercator_Auxiliary_Sphere",METHOD["Mercator_Auxiliary_Sphere"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",0.0],PARAMETER["Standard_Parallel_1",0.0],PARAMETER["Auxiliary_Sphere_Type",0.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["EPSG","3857"]]',
    preserve_shape="NO_PRESERVE_SHAPE",
    max_deviation=None,
    vertical="NO_VERTICAL"
)

# import historic wildifres into postgres
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=historic_fires_wgs84,
    Output_Geodatabase=sde_path
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde'>

In [123]:
cursor = conn.cursor()

with conn.transaction():
    #Create and Populate BIL Overlap Flag
    cursor.execute(f'''
        ALTER TABLE
            {schema}.{historic_fires_wgs84}
        ADD COLUMN IF NOT EXISTS
            bil_treatment_overlap
        BOOLEAN;
    ''')
    
    cursor.execute(f'''
         UPDATE 
             {schema}.{historic_fires_wgs84} h
         SET 
             bil_treatment_overlap = TRUE
         FROM 
             {schema}.{ti_stats_table} ti
         WHERE 
             ST_Intersects(ti.shape, h.shape) 
             AND  EXTRACT(YEAR FROM ti.treatment_date) < h.fire_year_int 
             AND ti.fund_source = 'BIL';  
    ''')

In [124]:
historic_fires_state_intersect_layer = 'historic_fires_cy_17_25_wgs84_intersect_states'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{historic_fires_wgs84}';'{sde_path}\{db}.{schema}.{states_fl_name}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{historic_fires_state_intersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.historic_fires_cy_17_25_wgs84_intersect_states'>

In [125]:
# Recent Wildifires Import

# Recent Wildfires URL
recent_fires_service_url = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters/FeatureServer/0"
recent_fires_fl_name = "recent_fires_cy_17_25"

recent_fires_query = "poly_PolygonDateTime BETWEEN date '2017-01-01' AND date '2025-12-31'"

arcpy.management.MakeFeatureLayer(recent_fires_service_url, recent_fires_fl_name, where_clause=recent_fires_query)

# import recent wildfires into postgres
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=recent_fires_fl_name,
    Output_Geodatabase=sde_path
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde'>

In [126]:
cursor = conn.cursor()

with conn.transaction():
    #Create and Populate BIL Overlap Flag
    cursor.execute(f'''
        ALTER TABLE
            {schema}.{recent_fires_fl_name}
        ADD COLUMN IF NOT EXISTS
            bil_treatment_overlap
        BOOLEAN;
    ''')
    
    cursor.execute(f'''
        UPDATE 
            {schema}.{recent_fires_fl_name} r
        SET 
            bil_treatment_overlap = TRUE
        FROM 
            {schema}.{ti_stats_table} ti
        WHERE 
           ST_Intersects(ti.shape, r.shape) 
           AND ti.treatment_date < r.poly_polygondatetime 
           AND ti.fund_source = 'BIL';
    ''')

In [127]:
recent_fires_state_intersect_layer = 'recent_fires_cy_17_25_intersect_states'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.recent_fires_cy_17_25';'{sde_path}\{db}.{schema}.{states_fl_name}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{recent_fires_state_intersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.recent_fires_cy_17_25_intersect_states'>

In [128]:
output_file = r'C:\Users\Ben Holloway\Documents\SWERI\5_year_report\state_stats_new.csv'

title = 'Treatment Stats by State and Fiscal Year'

query = f'''
    WITH treatment_index_data AS (
    SELECT 
        state_abbr as state_abbr,
        CASE 
            WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
            ELSE EXTRACT(YEAR FROM ti.treatment_date)
        END AS fiscal_year,
        COUNT(*) AS total_count,
        SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
        SUM(weighted_cost) as funding_amount
    FROM 
        {schema}.{ti_state_instersect_layer} ti
    WHERE 
        state_abbr IS NOT NULL
    GROUP BY 
        state_abbr, fiscal_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(ti.total_count, 0) AS total_count,
        COALESCE(ti.acres, 0) AS acres,
        COALESCE(ti.funding_amount, 0) AS funding_amount
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        treatment_index_data ti
    ON 
        s.state_abbr = ti.state_abbr
        AND s.year = ti.fiscal_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Treatment Stats by State and Fiscal Year


In [129]:
title = 'Treatment Stats by State and Fiscal Year (HIGH_COST error)'

query = f'''
    WITH treatment_index_data AS (
    SELECT 
        state_abbr as state_abbr,
        CASE 
            WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
            ELSE EXTRACT(YEAR FROM ti.treatment_date)
        END AS fiscal_year,
        SUM(weighted_cost) as funding_amount
    FROM 
        {schema}.{ti_state_instersect_layer} ti
    WHERE 
        state_abbr IS NOT NULL
    	AND 
    	error like '%HIGH_COST%'
    GROUP BY 
        state_abbr, fiscal_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(ti.funding_amount, 0) AS funding_amount
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        treatment_index_data ti
    ON 
        s.state_abbr = ti.state_abbr
        AND s.year = ti.fiscal_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Treatment Stats by State and Fiscal Year (HIGH_COST error)


In [130]:
title = '"BIL" Treatment Stats by State and Fiscal Year'

query = f'''
    WITH treatment_index_data AS (
    SELECT 
        state_abbr as state_abbr,
        CASE 
            WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
            ELSE EXTRACT(YEAR FROM ti.treatment_date)
        END AS fiscal_year,
        COUNT(*) AS total_count,
        SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
        SUM(weighted_cost) as funding_amount
    FROM 
        {schema}.{ti_state_instersect_layer} ti
    WHERE 
        state_abbr IS NOT NULL
    	and
    	fund_source = 'BIL'
    GROUP BY 
        state_abbr, fiscal_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(ti.total_count, 0) AS total_count,
        COALESCE(ti.acres, 0) AS acres,
        COALESCE(ti.funding_amount, 0) AS funding_amount
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        treatment_index_data ti
    ON 
        s.state_abbr = ti.state_abbr
        AND s.year = ti.fiscal_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: "BIL" Treatment Stats by State and Fiscal Year


In [131]:
title = '"BIL" Treatment Stats by State and Fiscal Year (HIGH_COST error)'

query = f'''
    WITH treatment_index_data AS (
    SELECT 
        state_abbr as state_abbr,
        CASE 
            WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
            ELSE EXTRACT(YEAR FROM ti.treatment_date)
        END AS fiscal_year,
        SUM(weighted_cost) as funding_amount
    FROM 
        {schema}.{ti_state_instersect_layer} ti
    WHERE 
        state_abbr IS NOT NULL
    	AND 
    	fund_source = 'BIL'
    	AND
    	error like '%HIGH_COST%'
    GROUP BY 
        state_abbr, fiscal_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(ti.funding_amount, 0) AS funding_amount
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        treatment_index_data ti
    ON 
        s.state_abbr = ti.state_abbr
        AND s.year = ti.fiscal_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: "BIL" Treatment Stats by State and Fiscal Year (HIGH_COST error)


In [132]:
title = 'Total Wildfire Count and Acres by State and Calendar Year (Historical Fires)'

query = f'''
    WITH historic_fire_data AS (
        SELECT 
            state_abbr as state_abbr,
            fire_year_int AS calendar_year,
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
        FROM 
            {schema}.{historic_fires_state_intersect_layer}
        WHERE 
            state_abbr IS NOT NULL
        GROUP BY 
            state_abbr , calendar_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(h.total_count, 0) AS total_count,
        COALESCE(h.acres, 0) AS acres
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        historic_fire_data h
    ON 
        s.state_abbr = h.state_abbr
        AND s.year = h.calendar_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total Wildfire Count and Acres by State and Calendar Year (Historical Fires)


In [133]:
title = 'Number of Wildfires that Intersct a "BIL" Treatment by State and Calendar Year (Historical Fires)'

query = f'''
    WITH historic_fire_data AS (
        SELECT 
            state_abbr as state_abbr,
            fire_year_int as calendar_year, 
            COUNT(*) AS total_count
        FROM 
            {schema}.{historic_fires_state_intersect_layer}
        WHERE 
            state_abbr IS NOT NULL
            AND
            bil_treatment_overlap = 1
        GROUP BY 
            state_abbr, calendar_year
    
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(h.total_count, 0) AS total_count
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        historic_fire_data h
    ON 
        s.state_abbr = h.state_abbr
        AND s.year = h.calendar_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Number of Wildfires that Intersct a "BIL" Treatment by State and Calendar Year (Historical Fires)


In [134]:
title = 'Total Wildfire Count and Acres by State and Calendar Year (Recent Fires)'

query = f'''
    WITH recent_fire_data AS (
        SELECT 
            state_abbr as state_abbr,
            EXTRACT(YEAR FROM recent_fires.poly_polygondatetime) as calendar_year, 
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
        FROM 
            {schema}.{recent_fires_state_intersect_layer} recent_fires
        WHERE 
            state_abbr IS NOT NULL
        GROUP BY 
            state_abbr, calendar_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(r.total_count, 0) AS total_count,
        COALESCE(r.acres, 0) AS acres
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        recent_fire_data r
    ON 
        s.state_abbr = r.state_abbr
        AND s.year = r.calendar_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total Wildfire Count and Acres by State and Calendar Year (Recent Fires)


In [135]:
title = 'Number of Wildfires that Intersct a "BIL" Treatment by State and Calendar Year (Recent Fires)'

query = f'''
    WITH recent_fire_data AS (
        SELECT 
            state_abbr as state_abbr,
            EXTRACT(YEAR FROM recent_fires.poly_polygondatetime) as calendar_year, 
            COUNT(*) AS total_count
        FROM 
            {schema}.{recent_fires_state_intersect_layer} recent_fires
        WHERE 
            state_abbr IS NOT NULL
            AND
            bil_treatment_overlap = 1
        GROUP BY 
            state_abbr, calendar_year
    )
    SELECT
        s.state_abbr AS state_abbr,
        s.year AS fiscal_year,
        COALESCE(r.total_count, 0) AS total_count
    FROM 
        {schema}.state_year_combinations s
    LEFT JOIN 
        recent_fire_data r
    ON 
        s.state_abbr = r.state_abbr
        AND s.year = r.calendar_year
    ORDER BY 
        s.state_abbr, s.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Number of Wildfires that Intersct a "BIL" Treatment by State and Calendar Year (Recent Fires)


State data is now done

In [136]:
# GACC Regions Import

gacc_service_url = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/DMP_NationalGACCBoundaries_Public/FeatureServer/0"

# Add GACC layers to the map
gacc_layer = active_map.addDataFromPath(gacc_service_url)
gacc_layer.name = 'gacc_regions'

gacc_projected = 'gacc_regions_wgs84'

#Reproject to WGS84 named gacc_regions_wgs84
arcpy.management.Project(
    in_dataset="gacc_regions",
    out_dataset=rf"C:\Users\Ben Holloway\Documents\SWERI\ArcPro\Sweri\Sweri.gdb\{gacc_projected}",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    transform_method="WGS_1984_(ITRF00)_To_NAD_1983",
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    preserve_shape="NO_PRESERVE_SHAPE",
    max_deviation=None,
    vertical="NO_VERTICAL"
)

# import gacc_regions_wgs84 into postgres
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=gacc_projected,
    Output_Geodatabase=sde_path
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde'>

In [137]:
ti_gacc_instersect_layer = 'treatment_index_fy_17_25_stats_intersect_gacc'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{ti_stats_table}';'{sde_path}\{db}.{schema}.{gacc_projected}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{ti_gacc_instersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.treatment_index_fy_17_25_stats_intersect_gacc'>

In [138]:
cursor = conn.cursor()

with conn.transaction():

    # Create and Populate Area Weight
    cursor.execute(f'''
        ALTER TABLE
            {schema}.{ti_gacc_instersect_layer}
        ADD COLUMN IF NOT EXISTS area_weight NUMERIC;
    ''')
    
    cursor.execute(f'''
        UPDATE {schema}.{ti_gacc_instersect_layer}
        SET area_weight = 1
            WHERE unique_id NOT IN (
                SELECT
                    t.unique_id
                FROM
                    {schema}.{ti_stats_table} t
                JOIN
                    {schema}.{gacc_projected} g
                ON
                    ST_Intersects(t.shape, g.shape)
                GROUP BY
                    t.unique_id
                HAVING
                    COUNT(DISTINCT g.gaccabbreviation) > 1
            );
    ''')
    
    cursor.execute(f'''
        UPDATE 
            {schema}.{ti_gacc_instersect_layer}
        SET 
            area_weight = (ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) / acres_calc
        WHERE 
            area_weight IS null;
    ''')

    # Use Area Weight to Caluculate Weighted Cost
    cursor.execute(f'''
        ALTER TABLE 
            {schema}.{ti_gacc_instersect_layer}
        ADD COLUMN IF NOT EXISTS
            weighted_cost NUMERIC;
''')

    cursor.execute(f'''
        UPDATE {schema}.{ti_gacc_instersect_layer}
        SET weighted_cost = total_cost * area_weight; 
    ''')

In [139]:
historic_fires_gacc_intersect_layer = 'historic_fires_cy_17_25_wgs84_intersect_gacc'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{historic_fires_wgs84}';'{sde_path}\{db}.{schema}.{gacc_projected}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{historic_fires_gacc_intersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.historic_fires_cy_17_25_wgs84_intersect_gacc'>

In [140]:
recent_fires_gacc_intersect_layer = 'recent_fires_cy_17_25_intersect_gacc'

arcpy.analysis.PairwiseIntersect(
    in_features=rf"'{sde_path}\{db}.{schema}.{recent_fires_fl_name}';'{sde_path}\{db}.{schema}.{gacc_projected}'",
    out_feature_class=rf"{sde_path}\{db}.{schema}.{recent_fires_gacc_intersect_layer}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT",
    precision="DEFAULT_PRECISION"
)

<Result 'C:\\Users\\Ben Holloway\\Documents\\SWERI\\ArcPro\\Sweri\\staging.sde\\sweri.staging.recent_fires_cy_17_25_intersect_gacc'>

In [141]:
output_file = rf'C:\Users\Ben Holloway\Documents\{db}\5_year_report\gacc_stats_new.csv'

title = 'Treatment Stats by GACC Region and Fiscal Year'

query = f'''
    WITH treatment_index_data AS (
        SELECT
            gaccabbreviation AS gacc_region,
            CASE
                WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
                ELSE EXTRACT(YEAR FROM ti.treatment_date)
            END AS fiscal_year,
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
            SUM(weighted_cost) AS funding_amount
        FROM
            {schema}.{ti_gacc_instersect_layer} ti
        WHERE
            gaccabbreviation IS NOT NULL
        GROUP BY
            gaccabbreviation, fiscal_year
    )
    SELECT 
        c.gacc_region_flag AS gacc_region_abbr,
        c.year AS fiscal_year,
        COALESCE(f.total_count, 0) AS total_count,
        COALESCE(f.acres, 0) AS acres,
        COALESCE(f.funding_amount, 0) AS funding_amount
    FROM 
        gacc_region_year_combinations c
    LEFT JOIN 
        treatment_index_data f
    ON 
        c.gacc_region_flag = f.gacc_region
        AND c.year = f.fiscal_year
    ORDER BY 
        c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Treatment Stats by GACC Region and Fiscal Year


In [142]:
title = 'Treatment Stats by GACC Region and Fiscal Year (HIGH_COST error)'

query = f'''
    WITH treatment_index_data AS (
        SELECT
            gaccabbreviation AS gacc_region,
            CASE
                WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
                ELSE EXTRACT(YEAR FROM ti.treatment_date)
            END AS fiscal_year,
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
            SUM(weighted_cost) AS funding_amount
        FROM
            {schema}.{ti_gacc_instersect_layer} ti
        WHERE
            gaccabbreviation IS NOT NULL
    		and
    		error like '%HIGH_COST%'
        GROUP BY
            gaccabbreviation, fiscal_year
    )
    SELECT 
        c.gacc_region_flag AS gacc_region_abbr,
        c.year AS fiscal_year,
        COALESCE(f.total_count, 0) AS total_count,
        COALESCE(f.acres, 0) AS acres,
        COALESCE(f.funding_amount, 0) AS funding_amount
    FROM 
        gacc_region_year_combinations c
    LEFT JOIN 
        treatment_index_data f
    ON 
        c.gacc_region_flag = f.gacc_region
        AND c.year = f.fiscal_year
    ORDER BY 
        c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Treatment Stats by GACC Region and Fiscal Year (HIGH_COST error)


In [143]:
title = '"BIL" Treatment Stats by GACC Region and Fiscal Year'

query = f'''
    WITH treatment_index_data AS (
        SELECT
            gaccabbreviation AS gacc_region,
            CASE
                WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
                ELSE EXTRACT(YEAR FROM ti.treatment_date)
            END AS fiscal_year,
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
            SUM(weighted_cost) AS funding_amount
        FROM
            {schema}.{ti_gacc_instersect_layer} ti
        WHERE
            gaccabbreviation IS NOT NULL
    	and
    	fund_source = 'BIL'
        GROUP BY
            gaccabbreviation, fiscal_year
    )
    SELECT 
        c.gacc_region_flag AS gacc_region_abbr,
        c.year AS fiscal_year,
        COALESCE(f.total_count, 0) AS total_count,
        COALESCE(f.acres, 0) AS acres,
        COALESCE(f.funding_amount, 0) AS funding_amount
    FROM 
        gacc_region_year_combinations c
    LEFT JOIN 
        treatment_index_data f
    ON 
        c.gacc_region_flag = f.gacc_region
        AND c.year = f.fiscal_year
    ORDER BY 
        c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: "BIL" Treatment Stats by GACC Region and Fiscal Year


In [144]:
title = '"BIL" Treatment Stats by GACC Region and Fiscal Year (HIGH_COST error)'

query = f'''
    WITH treatment_index_data AS (
        SELECT
            gaccabbreviation AS gacc_region,
            CASE
                WHEN EXTRACT(MONTH FROM ti.treatment_date) >= 10 THEN EXTRACT(YEAR FROM ti.treatment_date) + 1
                ELSE EXTRACT(YEAR FROM ti.treatment_date)
            END AS fiscal_year,
            COUNT(*) AS total_count,
            SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres,
            SUM(weighted_cost) AS funding_amount
        FROM
            {schema}.{ti_gacc_instersect_layer} ti
        WHERE
            gaccabbreviation IS NOT NULL
    	AND
    		fund_source = 'BIL'
    	AND
    		error like '%HIGH_COST%'
        GROUP BY
            gaccabbreviation, fiscal_year
    )
    SELECT 
        c.gacc_region_flag AS gacc_region_abbr,
        c.year AS fiscal_year,
        COALESCE(f.total_count, 0) AS total_count,
        COALESCE(f.acres, 0) AS acres,
        COALESCE(f.funding_amount, 0) AS funding_amount
    FROM 
        gacc_region_year_combinations c
    LEFT JOIN 
        treatment_index_data f
    ON 
        c.gacc_region_flag = f.gacc_region
        AND c.year = f.fiscal_year
    ORDER BY 
        c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: "BIL" Treatment Stats by GACC Region and Fiscal Year (HIGH_COST error)


In [145]:
title = 'Total Wildfire Count and Acres by GACC Region and Calendar Year (Historical Fires)'

query = f'''
WITH historic_fire_data AS (
    SELECT 
			gaccabbreviation AS gacc_region,
			fire_year_int AS calendar_year,
			COUNT(*) AS total_count,
			SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM 
        {schema}.{historic_fires_gacc_intersect_layer}
    WHERE 
        gaccabbreviation IS NOT NULL
    GROUP BY 
        gaccabbreviation , calendar_year
)
SELECT 
    c.gacc_region_flag AS gacc_region_abbr,
    c.year AS fiscal_year,
    COALESCE(f.total_count, 0) AS total_count,
    COALESCE(f.acres, 0) AS acres
FROM 
    gacc_region_year_combinations c
LEFT JOIN 
    historic_fire_data f
ON 
    c.gacc_region_flag = f.gacc_region
    AND c.year = f.calendar_year
ORDER BY 
    c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total Wildfire Count and Acres by GACC Region and Calendar Year (Historical Fires)


In [146]:
title = 'Total BIL Overlap Wildfire Count by GACC Region and Calendar Year (Historical Fires)'

query = f'''
WITH historic_fire_data AS (
    SELECT 
		gaccabbreviation AS gacc_region,
		fire_year_int AS calendar_year,
		COUNT(*) AS total_count
    FROM 
        {schema}.{historic_fires_gacc_intersect_layer}
    WHERE 
        gaccabbreviation IS NOT NULL
		and
		bil_treatment_overlap = 1
    GROUP BY 
        gaccabbreviation , calendar_year
)
SELECT 
    c.gacc_region_flag AS gacc_region_abbr,
    c.year AS fiscal_year,
    COALESCE(f.total_count, 0) AS total_count
FROM 
    gacc_region_year_combinations c
LEFT JOIN 
    historic_fire_data f
ON 
    c.gacc_region_flag = f.gacc_region
    AND c.year = f.calendar_year
ORDER BY 
    c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total BIL Overlap Wildfire Count by GACC Region and Calendar Year (Historical Fires)


In [147]:
title = 'Total Wildfire Count and Acres by GACC Region and Calendar Year (Recent Fires)'

query = f'''
WITH recent_fire_data AS (
    SELECT 
			gaccabbreviation AS gacc_region,
			EXTRACT(YEAR FROM recent_fires.poly_polygondatetime) as calendar_year, 
			COUNT(*) AS total_count,
			SUM(ST_Area(ST_Transform(shape, 5070)) / 4046.8564224) AS acres
    FROM 
        {schema}.{recent_fires_gacc_intersect_layer} recent_fires
    WHERE 
        gaccabbreviation IS NOT NULL
    GROUP BY 
        gaccabbreviation , calendar_year
)
SELECT 
    c.gacc_region_flag AS gacc_region_abbr,
    c.year AS fiscal_year,
    COALESCE(r.total_count, 0) AS total_count,
    COALESCE(r.acres, 0) AS acres
FROM 
    gacc_region_year_combinations c
LEFT JOIN 
    recent_fire_data r
ON 
    c.gacc_region_flag = r.gacc_region
    AND c.year = r.calendar_year
ORDER BY 
    c.gacc_region_flag, c.year;

'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total Wildfire Count and Acres by GACC Region and Calendar Year (Recent Fires)


In [148]:
title = 'Total BIL Overlap Wildfire Count by GACC Region and Calendar Year (Recent Fires)'

query = f'''
WITH recent_fire_data AS (
    SELECT 
			gaccabbreviation AS gacc_region,
			EXTRACT(YEAR FROM recent_fires.poly_polygondatetime) as calendar_year, 
			COUNT(*) AS total_count
    FROM 
        {schema}.{recent_fires_gacc_intersect_layer} recent_fires
    WHERE 
        gaccabbreviation IS NOT NULL
	AND
		bil_treatment_overlap = 1
    GROUP BY 
        gaccabbreviation , calendar_year
)
SELECT 
    c.gacc_region_flag AS gacc_region_abbr,
    c.year AS fiscal_year,
    COALESCE(r.total_count, 0) AS total_count
FROM 
    gacc_region_year_combinations c
LEFT JOIN 
    recent_fire_data r
ON 
    c.gacc_region_flag = r.gacc_region
    AND c.year = r.calendar_year
ORDER BY 
    c.gacc_region_flag, c.year;
'''

execute_and_append_to_csv(conn, query, output_file, title)

Data successfully appended with section: Total BIL Overlap Wildfire Count by GACC Region and Calendar Year (Recent Fires)
